In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_excel('../CSVs/00_raw_mozambique_data.xlsx')

In [3]:
df.columns

Index(['Village Code', 'Month of VISIT DATE', 'Study Month',
       'Actual.Spray Status', 'u5.RDT.pos', 'u5.RDT.neg', 'Residents.u5',
       'o5.RDT.pos', 'o5.RDT.neg', 'Residents.o5', 'Total.RDT.pos',
       'Total.RDT.Neg', 'Total.Residents', 'u5 Incidence', 'Total Incidence',
       'Clinic', 'Inpatient', 'APE', 'ANC', 'Maternity', 'Other', 'Total.RDTs',
       'Total.RDTs.u5', 'Percent.RDTs.u5', 'Percent.RDTs.Clinic',
       'Percent.RDTs.Inpatient', 'Percent.RDTs.APE', 'Percent.RDTs.ANC',
       'Percent.RDTs.Maternity', 'Percent.RDTs.Other', 'Male', 'Female',
       'Percent.RDTs.Male', 'Avg. Km To Closest Health Facility', 'Total.TPR',
       'u5.TPR'],
      dtype='object')

In [4]:
df.head()

,Village Code,Month of VISIT DATE,Study Month,Actual.Spray Status,u5.RDT.pos,u5.RDT.neg,Residents.u5,o5.RDT.pos,o5.RDT.neg,Residents.o5,...,Percent.RDTs.APE,Percent.RDTs.ANC,Percent.RDTs.Maternity,Percent.RDTs.Other,Male,Female,Percent.RDTs.Male,Avg. Km To Closest Health Facility,Total.TPR,u5.TPR
0,7001,2016-12-01,0,1,19.0,17.0,287,15,15,986,...,0.257576,0.0,0.0,0.0,35,31,0.530303,NaN,0.515152,0.527778
1,7002,2016-12-01,0,1,NaN,1.0,319,0,0,1181,...,0.000000,0.0,0.0,0.0,0,1,0.000000,7.6,0.000000,0.000000
2,7003,2016-12-01,0,0,15.0,13.0,686,10,16,2945,...,0.000000,0.0,0.0,0.0,24,30,0.444444,1.8,0.462963,0.535714
3,7004,2016-12-01,0,1,39.0,56.0,1073,92,62,4119,...,0.000000,0.0,0.0,0.0,120,129,0.481928,0.7,0.526104,0.410526
4,7005,2016-12-01,0,1,6.0,18.0,500,11,11,1949,...,0.000000,0.0,0.0,0.0,22,24,0.478261,0.9,0.369565,0.250000


In [5]:
df.columns = [
    'village_code', 'visit_month', 'study_month', 'spray_status',
    'u5_rdt_positive', 'u5_rdt_negative', 'u5_residents',
    'o5_rdt_positive', 'o5_rdt_negative', 'o5_residents',
    'total_rdt_positive', 'total_rdt_negative', 'total_residents',
    'u5_incidence', 'total_incidence',
    'clinic_tests', 'inpatient_tests', 'ape_tests', 'anc_tests', 'maternity_tests', 'other_tests',
    'total_rdts', 'u5_total_rdts', 'u5_rdt_percent',
    'clinic_rdt_percent', 'inpatient_rdt_percent', 'ape_rdt_percent',
    'anc_rdt_percent', 'maternity_rdt_percent', 'other_rdt_percent',
    'male_tests', 'female_tests', 'male_rdt_percent',
    'avg_distance_to_health_facility_km', 'total_tpr', 'u5_tpr'
]
avg_dist = 'avg_distance_to_health_facility_km' # Just shortening the parameter name
# Let's sort by visit_month
df.sort_values(['visit_month', 'village_code'], inplace=True)
df.describe()

,village_code,visit_month,study_month,spray_status,u5_rdt_positive,u5_rdt_negative,u5_residents,o5_rdt_positive,o5_rdt_negative,o5_residents,...,ape_rdt_percent,anc_rdt_percent,maternity_rdt_percent,other_rdt_percent,male_tests,female_tests,male_rdt_percent,avg_distance_to_health_facility_km,total_tpr,u5_tpr
count,3963.00000,3963,3963.000000,3963.000000,3647.000000,3186.000000,3963.000000,3963.000000,3963.000000,3963.000000,...,3963.000000,3963.000000,3963.000000,3963.000000,3963.000000,3963.000000,3963.000000,2321.000000,3963.000000,3780.000000
mean,7113.27353,2017-11-05 07:30:55.866767616,11.163260,0.432501,13.244585,8.301004,139.498864,13.550341,11.525107,613.789806,...,0.163426,0.000840,0.000553,0.000634,20.290184,23.647237,0.462577,6.937096,0.605767,0.669820
min,7001.00000,2016-12-01 00:00:00,0.000000,0.000000,1.000000,1.000000,9.000000,0.000000,0.000000,38.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.100000,0.000000,0.000000
25%,7050.00000,2017-06-01 00:00:00,6.000000,0.000000,3.000000,2.000000,57.000000,2.000000,2.000000,262.000000,...,0.000000,0.000000,0.000000,0.000000,4.000000,5.000000,0.380952,2.800000,0.500000,0.500000
50%,7112.00000,2017-11-01 00:00:00,11.000000,0.000000,8.000000,4.000000,92.000000,6.000000,5.000000,408.000000,...,0.000000,0.000000,0.000000,0.000000,11.000000,12.000000,0.461538,6.000000,0.609756,0.687500
75%,7177.00000,2018-05-01 00:00:00,17.000000,1.000000,17.000000,10.000000,183.000000,15.000000,12.000000,745.000000,...,0.169576,0.000000,0.000000,0.000000,24.000000,28.000000,0.538462,9.900000,0.739130,0.846154
max,7240.00000,2018-10-01 00:00:00,22.000000,1.000000,287.000000,174.000000,1073.000000,429.000000,259.000000,4367.000000,...,1.000000,1.000000,1.000000,0.333333,534.000000,570.000000,1.000000,30.100000,1.000000,1.000000
std,70.84040,NaN,6.579675,0.495485,16.768501,12.066071,135.287016,23.587544,19.928863,606.322245,...,0.303537,0.017073,0.016592,0.010134,31.329985,35.758984,0.176193,5.574070,0.205522,0.242955


In [6]:
df['incidence_per_1000'] = round((df['total_incidence'] / df['total_residents']) * 1000, 2)
# Very important feature

In [7]:
df[['total_residents', 'total_rdt_positive', 'total_rdts', 'total_tpr', 'incidence_per_1000']]

,total_residents,total_rdt_positive,total_rdts,total_tpr,incidence_per_1000
0,1273,34,66,0.515152,20.98
1,1500,0,1,0.000000,0.00
2,3631,25,54,0.462963,1.90
3,5192,131,249,0.526104,4.86
4,2449,17,46,0.369565,2.83
...,...,...,...,...,...
3958,361,11,13,0.846154,84.41
3959,341,15,37,0.405405,129.00
3960,530,1,4,0.250000,3.56
3961,249,5,13,0.384615,80.64


In [8]:
print(f"""\
Unique distances from health facilities: {len(df[avg_dist].unique())} / {len(df[avg_dist])}
Number of villages: {len(df['village_code'].unique())}
NaN values for hospital distance: {len(df[df[avg_dist].isna()])} / {len(df[avg_dist])}
    """)
# This is great, this means that the data was found from the same place that was checked regularly, good to avoid variable circumstances
# Problem is the number of NaN values in the paramter, clipping it leads to 40% data loss
# Also, there are way more villages than unique distances. That could happen though.

Unique distances from health facilities: 82 / 3963
Number of villages: 188
NaN values for hospital distance: 1642 / 3963
    


In [9]:
df[df[avg_dist].isna()]['incidence_per_1000'].describe()

count    1642.000000
mean      158.042674
std       346.216103
min         0.000000
25%        28.592500
50%        72.090000
75%       143.240000
max      4058.110000
Name: incidence_per_1000, dtype: float64

In [10]:
df[df[avg_dist].notna()]['incidence_per_1000'].describe()
# Rows with NaN values for average distances have higher incidence than those with values
# This might mean: "Facilities are too far (NaN distance) and incidences are higher due to isolation from hospital"
# Or it might be a coincidence. The incidence difference is only by 7 on average, after all.

count    2321.000000
mean       74.540422
std       125.105873
min         0.000000
25%        11.840000
50%        30.280000
75%        83.020000
max      1487.800000
Name: incidence_per_1000, dtype: float64

In [11]:
df.loc[df['visit_month'] == df['visit_month'].unique()[0]]['village_code']
# This dataset is better than I expected, very granular, they visited every village for each visit day

0      7001
1      7002
2      7003
3      7004
4      7005
       ... 
153    7232
154    7233
155    7236
156    7238
157    7239
Name: village_code, Length: 158, dtype: int64

In [12]:
df[df['u5_rdt_positive'].isna()]
# This is interesting, there aren't instances of 0 but there are 316 instances of it equaling NaN, maybe I should set them to zeros

,village_code,visit_month,study_month,spray_status,u5_rdt_positive,u5_rdt_negative,u5_residents,o5_rdt_positive,o5_rdt_negative,o5_residents,...,anc_rdt_percent,maternity_rdt_percent,other_rdt_percent,male_tests,female_tests,male_rdt_percent,avg_distance_to_health_facility_km,total_tpr,u5_tpr,incidence_per_1000
1,7002,2016-12-01,0,1,NaN,1.0,319,0,0,1181,...,0.0,0.0,0.0,0,1,0.00,7.6,0.00,0.0,0.00
10,7012,2016-12-01,0,1,NaN,NaN,197,0,1,649,...,0.0,0.0,0.0,0,1,0.00,11.0,0.00,NaN,0.00
14,7017,2016-12-01,0,1,NaN,1.0,171,0,1,856,...,0.0,0.0,0.0,1,1,0.50,11.8,0.00,0.0,0.00
20,7024,2016-12-01,0,1,NaN,3.0,94,1,1,382,...,0.0,0.0,0.0,2,3,0.40,10.4,0.20,0.0,4.41
21,7025,2016-12-01,0,1,NaN,NaN,111,0,1,450,...,0.0,0.0,0.0,0,1,0.00,4.5,0.00,NaN,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3941,7209,2018-10-01,22,0,NaN,NaN,40,1,0,294,...,0.0,0.0,0.0,0,1,0.00,NaN,1.00,NaN,8.96
3942,7210,2018-10-01,22,0,NaN,NaN,53,2,2,295,...,0.0,0.0,0.0,0,4,0.00,NaN,0.50,NaN,16.51
3946,7217,2018-10-01,22,1,NaN,1.0,36,0,0,236,...,0.0,0.0,0.0,1,0,1.00,NaN,0.00,0.0,0.00
3957,7233,2018-10-01,22,1,NaN,1.0,74,3,0,388,...,0.0,0.0,0.0,2,2,0.50,5.3,0.75,0.0,14.06


In [13]:
# Let's check if they have low total incidences in kind
print(f"Incidence on NaNs (possible zeros): {np.mean(df[df['u5_rdt_positive'].isna()]['incidence_per_1000'])}")
print(f"Incidence on non-NaNs: {df[df['u5_rdt_positive'].notna()].describe()['incidence_per_1000']['mean']}")
# The ones with NaNs have incidences 10x less than those that don't. That's good.

Incidence on NaNs (possible zeros): 16.714082278481012
Incidence on non-NaNs: 117.14635042500687


In [14]:
print(f"Number of u5_rdt_negative NaNs: {len(df[df['u5_rdt_negative'].isna()]['incidence_per_1000'])}")
print(f"Mean of incidences in these NaNs: {np.mean(df[df['u5_rdt_negative'].isna()]['incidence_per_1000'].describe()['mean'])}")
print(f"Mean of incidences that aren't NaNs: {np.mean(df[df['u5_rdt_negative'].notna()]['incidence_per_1000'].describe()['mean'])}")
# Hmm, the negative ones have NaNs too, but their corresponding total_incidences are not high...
# I can't rely on NaN = 0. Maybe only on the ones for the negative counts.
# I should drop this column at the least

Number of u5_rdt_negative NaNs: 777
Mean of incidences in these NaNs: 48.74290862290862
Mean of incidences that aren't NaNs: 123.86727871939735


In [15]:
df.drop(columns=[text for text in df.columns if re.search(r'\b\w+_rdt_percent', text)], inplace=True)

In [16]:
df = df[df['total_incidence'] < df['total_residents']]
# There are a considerable amount of instances where total_incidence > total_residents. Which makes no sense.
# So I am removing those instances

In [17]:
df['incidence_per_1000'].describe()

count    3914.000000
mean       87.192563
std       124.512564
min         0.000000
25%        15.750000
50%        44.440000
75%       107.410000
max       987.650000
Name: incidence_per_1000, dtype: float64

In [18]:
df1 = df[df['u5_residents'] < df['u5_incidence']][['u5_residents', 'u5_incidence']]
df1['diff'] = df['u5_incidence'] - df['u5_residents']
df1.describe()['diff']
# This is bad

count    1552.000000
mean      108.387858
std       130.150898
min         0.178082
25%        32.238095
50%        68.909091
75%       135.982456
max      1498.019608
Name: diff, dtype: float64

In [19]:
# I have to drop everything with under age 5 info
# I intend on trying to clip and work with them later
df.drop(columns=[text for text in df.columns if re.search(r'u5_+', text)], inplace=True)
df.columns

Index(['village_code', 'visit_month', 'study_month', 'spray_status',
       'o5_rdt_positive', 'o5_rdt_negative', 'o5_residents',
       'total_rdt_positive', 'total_rdt_negative', 'total_residents',
       'total_incidence', 'clinic_tests', 'inpatient_tests', 'ape_tests',
       'anc_tests', 'maternity_tests', 'other_tests', 'total_rdts',
       'male_tests', 'female_tests', 'avg_distance_to_health_facility_km',
       'total_tpr', 'incidence_per_1000'],
      dtype='object')

In [20]:
train = df[df['visit_month'] < '2018-01']
test = df[df['visit_month'] >= '2018-01']

In [21]:
df['positivity_rate'] = round(df['total_rdt_positive'] / df['total_rdts'], 2)

In [22]:
print(np.mean(df[df['spray_status'] == 1]['incidence_per_1000'].tolist()))
print(np.mean(df[df['spray_status'] == 0]['incidence_per_1000'].tolist()))
# There are considerably more cases when the villages aren't sprayed

75.79995913601869
96.05922762380736


In [23]:
print(np.mean(df[df['spray_status'] == 1]['positivity_rate'].tolist()))
print(np.mean(df[df['spray_status'] == 0]['positivity_rate'].tolist()))
# Spray status isn't a of 'positivity_rate'. Let's define them

0.5924868651488616
0.6152975920036347


In [24]:
nasa = pd.read_csv('../CSVs/05_climate_data.csv')

In [25]:
rain = nasa[nasa['PARAMETER'] == 'PRECTOTCORR'].drop(columns=['ANN', 'PARAMETER'])
rain

,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC
0,2016,6.52,2.65,5.03,2.61,1.09,0.74,2.13,0.21,0.01,0.19,1.55,4.39
13,2017,5.64,3.42,5.78,4.80,0.57,1.12,0.82,0.36,0.18,0.34,6.41,10.29
21,2018,9.30,5.32,3.17,4.29,0.62,0.61,4.00,0.23,0.36,0.56,2.74,3.64


In [26]:
# Melt converts wide columns (JAN, FEB, …) into rows
rain_long = rain.melt(id_vars='YEAR', 
                      var_name='month_str', 
                      value_name='rainfall')

In [27]:
rain.head()

,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC
0,2016,6.52,2.65,5.03,2.61,1.09,0.74,2.13,0.21,0.01,0.19,1.55,4.39
13,2017,5.64,3.42,5.78,4.80,0.57,1.12,0.82,0.36,0.18,0.34,6.41,10.29
21,2018,9.30,5.32,3.17,4.29,0.62,0.61,4.00,0.23,0.36,0.56,2.74,3.64


In [28]:
new_list = []
for x in df['visit_month'].tolist():
    new_list.append(x.day)

In [29]:
month_map = {
    'JAN':1, 'FEB':2, 'MAR':3, 'APR':4, 'MAY':5, 'JUN':6,
    'JUL':7, 'AUG':8, 'SEP':9, 'OCT':10, 'NOV':11, 'DEC':12
}
rain_long['month_num'] = rain_long['month_str'].map(month_map)

rain_long['visit_month'] = pd.to_datetime(
    rain_long['YEAR'].astype(str) + '-' + rain_long['month_num'].astype(str) + '-01'
)
rain_long = rain_long[['visit_month','rainfall']]

In [30]:
rain_long.head()

,visit_month,rainfall
0,2016-01-01,6.52
1,2017-01-01,5.64
2,2018-01-01,9.30
3,2016-02-01,2.65
4,2017-02-01,3.42


In [31]:
df = df.merge(rain_long, on='visit_month', how='left')

In [32]:
df = df.sort_values(['village_code', 'visit_month'])
df['prev_month_rainfall'] = df.groupby('village_code')['rainfall'].shift(1)

In [33]:
df.head(len(df['visit_month'].unique().tolist()))

,village_code,visit_month,study_month,spray_status,o5_rdt_positive,o5_rdt_negative,o5_residents,total_rdt_positive,total_rdt_negative,total_residents,...,other_tests,total_rdts,male_tests,female_tests,avg_distance_to_health_facility_km,total_tpr,incidence_per_1000,positivity_rate,rainfall,prev_month_rainfall
0,7001,2016-12-01,0,1,15,15,986,34,32,1273,...,0.0,66,35,31,NaN,0.515152,20.98,0.52,4.39,NaN
158,7001,2017-01-01,1,1,34,10,986,78,27,1273,...,0.0,105,49,56,NaN,0.742857,48.13,0.74,5.64,4.39
329,7001,2017-02-01,2,1,27,7,986,76,29,1273,...,0.0,105,59,46,NaN,0.723810,46.90,0.72,3.42,5.64
492,7001,2017-03-01,3,1,18,3,986,33,16,1273,...,0.0,49,26,23,NaN,0.673469,20.36,0.67,5.78,3.42
650,7001,2017-04-01,4,1,30,11,986,77,24,1273,...,NaN,101,50,51,NaN,0.762376,47.52,0.76,4.80,5.78
811,7001,2017-05-01,5,1,21,2,986,58,7,1273,...,0.0,65,27,38,NaN,0.892308,35.79,0.89,0.57,4.80
976,7001,2017-06-01,6,1,26,9,986,94,22,1273,...,0.0,116,58,58,NaN,0.810345,58.01,0.81,1.12,0.57
1139,7001,2017-07-01,7,1,22,18,986,62,50,1273,...,0.0,112,58,54,NaN,0.553571,38.26,0.55,0.82,1.12
1311,7001,2017-08-01,8,1,55,16,986,96,37,1273,...,0.0,133,68,65,NaN,0.721805,59.24,0.72,0.36,0.82
1482,7001,2017-09-01,9,1,27,12,986,56,22,1273,...,0.0,78,43,35,NaN,0.717949,34.56,0.72,0.18,0.36


In [34]:
df[df['rainfall'] == 4.39]['visit_month'].unique()

<DatetimeArray>
['2016-12-01 00:00:00']
Length: 1, dtype: datetime64[ns]

In [35]:
a = df[['village_code', avg_dist]][df[avg_dist].isna()]['village_code'].unique()

In [36]:
b = df[['village_code', avg_dist]][df[avg_dist].notna()]['village_code'].unique()

In [37]:
def no_common_elements(a, b):
    a_flat = a.ravel()
    b_flat = b.ravel()
    
    common = np.intersect1d(a_flat, b_flat)
    return len(common) == 0 
no_common_elements(a, b)
# This means distance is missing for specific village codes. There isn't a village marked with a value somewhere and NaN some other time
# This rules out imputing

True

In [38]:
# Imputing isn't generally a good idea because there are specific villages that aren't assigned a value.
# Regardless, however, I am gonna impute it and assign a flag saying that the distance is missing.
df['dist_missing'] = df[avg_dist].isna().astype(int)

In [39]:
df.head()

,village_code,visit_month,study_month,spray_status,o5_rdt_positive,o5_rdt_negative,o5_residents,total_rdt_positive,total_rdt_negative,total_residents,...,total_rdts,male_tests,female_tests,avg_distance_to_health_facility_km,total_tpr,incidence_per_1000,positivity_rate,rainfall,prev_month_rainfall,dist_missing
0,7001,2016-12-01,0,1,15,15,986,34,32,1273,...,66,35,31,NaN,0.515152,20.98,0.52,4.39,NaN,1
158,7001,2017-01-01,1,1,34,10,986,78,27,1273,...,105,49,56,NaN,0.742857,48.13,0.74,5.64,4.39,1
329,7001,2017-02-01,2,1,27,7,986,76,29,1273,...,105,59,46,NaN,0.723810,46.90,0.72,3.42,5.64,1
492,7001,2017-03-01,3,1,18,3,986,33,16,1273,...,49,26,23,NaN,0.673469,20.36,0.67,5.78,3.42,1
650,7001,2017-04-01,4,1,30,11,986,77,24,1273,...,101,50,51,NaN,0.762376,47.52,0.76,4.80,5.78,1


In [40]:
df[avg_dist] = df[avg_dist].fillna(7)
df[df[avg_dist] == 7]

,village_code,visit_month,study_month,spray_status,o5_rdt_positive,o5_rdt_negative,o5_residents,total_rdt_positive,total_rdt_negative,total_residents,...,total_rdts,male_tests,female_tests,avg_distance_to_health_facility_km,total_tpr,incidence_per_1000,positivity_rate,rainfall,prev_month_rainfall,dist_missing
0,7001,2016-12-01,0,1,15,15,986,34,32,1273,...,66,35,31,7.0,0.515152,20.98,0.52,4.39,NaN,1
158,7001,2017-01-01,1,1,34,10,986,78,27,1273,...,105,49,56,7.0,0.742857,48.13,0.74,5.64,4.39,1
329,7001,2017-02-01,2,1,27,7,986,76,29,1273,...,105,59,46,7.0,0.723810,46.90,0.72,3.42,5.64,1
492,7001,2017-03-01,3,1,18,3,986,33,16,1273,...,49,26,23,7.0,0.673469,20.36,0.67,5.78,3.42,1
650,7001,2017-04-01,4,1,30,11,986,77,24,1273,...,101,50,51,7.0,0.762376,47.52,0.76,4.80,5.78,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
975,7240,2017-05-01,5,0,0,0,148,1,0,191,...,1,0,1,7.0,1.000000,27.41,1.00,0.57,NaN,1
1138,7240,2017-06-01,6,0,1,0,148,4,0,191,...,4,3,1,7.0,1.000000,109.65,1.00,1.12,0.57,1
1481,7240,2017-08-01,8,0,0,0,148,1,0,191,...,1,1,0,7.0,1.000000,27.41,1.00,0.36,1.12,1
1831,7240,2017-10-01,10,0,1,0,148,1,0,191,...,1,1,0,7.0,1.000000,27.41,1.00,0.34,0.36,1


In [41]:
df.columns

Index(['village_code', 'visit_month', 'study_month', 'spray_status',
       'o5_rdt_positive', 'o5_rdt_negative', 'o5_residents',
       'total_rdt_positive', 'total_rdt_negative', 'total_residents',
       'total_incidence', 'clinic_tests', 'inpatient_tests', 'ape_tests',
       'anc_tests', 'maternity_tests', 'other_tests', 'total_rdts',
       'male_tests', 'female_tests', 'avg_distance_to_health_facility_km',
       'total_tpr', 'incidence_per_1000', 'positivity_rate', 'rainfall',
       'prev_month_rainfall', 'dist_missing'],
      dtype='object')

In [42]:
df.drop(columns=['o5_rdt_positive', 'o5_rdt_negative', 'o5_residents', 'total_rdt_positive', 'total_rdt_negative',
                'ape_tests', 'anc_tests', 'male_tests', 'female_tests', 'other_tests', 'maternity_tests', 'inpatient_tests', 'study_month',
                'total_tpr', 'total_incidence'],
        inplace=True)

In [43]:
df[df['spray_status'] == 1].describe()['incidence_per_1000']

count    1713.000000
mean       75.799959
min         0.000000
25%        11.800000
50%        33.530000
75%        93.050000
max       862.920000
std       109.568234
Name: incidence_per_1000, dtype: float64

In [44]:
df[df['spray_status'] == 0].describe()['incidence_per_1000']

count    2201.000000
mean       96.059228
min         0.000000
25%        20.270000
50%        51.630000
75%       115.000000
max       987.650000
std       134.361201
Name: incidence_per_1000, dtype: float64

In [45]:
df['total_rdts'].describe()

count    3914.000000
mean       43.681911
std        66.633335
min         1.000000
25%         9.000000
50%        22.000000
75%        51.000000
max      1104.000000
Name: total_rdts, dtype: float64

In [46]:
df[['clinic_tests', 'total_rdts']]

,clinic_tests,total_rdts
0,0.0,66
158,0.0,105
329,0.0,105
492,0.0,49
650,NaN,101
...,...,...
975,NaN,1
1138,4.0,4
1481,0.0,1
1831,0.0,1


In [47]:
np.intersect1d(df[df['spray_status'] == 0]['village_code'].unique(), df[df['spray_status'] == 1]['village_code'].unique())
# Another interesting observation is that there isn't a village that was sprayed at a certain time and not sprayed at another
# This is not very good because it doesn't show diversity WITHIN a village to check how much spraying really makes a difference within a region

array([], dtype=int64)

In [48]:
df.columns

Index(['village_code', 'visit_month', 'spray_status', 'total_residents',
       'clinic_tests', 'total_rdts', 'avg_distance_to_health_facility_km',
       'incidence_per_1000', 'positivity_rate', 'rainfall',
       'prev_month_rainfall', 'dist_missing'],
      dtype='object')

In [49]:
df['clinic_tests'] = df['clinic_tests'].fillna(df['clinic_tests'].median())

In [50]:
nulls = []
for x in df.columns:
    nulls.append(np.sum(df[x].isna().tolist()))
nulls
df.columns[-2]
# One more NaN to handle

'prev_month_rainfall'

In [51]:
nan_containers = df[df['prev_month_rainfall'].isna()]['visit_month'].unique()
nan_containers

<DatetimeArray>
['2016-12-01 00:00:00', '2017-01-01 00:00:00', '2017-06-01 00:00:00',
 '2017-12-01 00:00:00', '2017-02-01 00:00:00', '2017-05-01 00:00:00',
 '2017-08-01 00:00:00', '2017-03-01 00:00:00', '2017-04-01 00:00:00']
Length: 9, dtype: datetime64[ns]

In [52]:
df[df['visit_month'] == nan_containers[8]]['prev_month_rainfall']

650    5.78
651    5.78
652    5.78
653    5.78
654    5.78
       ... 
806    3.42
807    5.78
808    5.78
809    5.78
810    5.78
Name: prev_month_rainfall, Length: 161, dtype: float64

In [53]:
extra = pd.read_csv('../CSVs/02_lagged_rainfall_data.csv')
extra.drop(columns=['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'ANN'], inplace=True)

In [54]:
extra.drop(index=[0, 3, 4, 5, 6, 7], inplace=True)

In [55]:
extra

,PARAMETER,YEAR,DEC
1,PRECTOTCORR,2015,3.58
2,PRECTOTCORR,2016,4.39


In [56]:
df[df['prev_month_rainfall'].isna()][['prev_month_rainfall', 'visit_month']].sort_values('visit_month')

,prev_month_rainfall,visit_month
0,NaN,2016-12-01
97,NaN,2016-12-01
98,NaN,2016-12-01
99,NaN,2016-12-01
100,NaN,2016-12-01
...,...,...
990,NaN,2017-06-01
1069,NaN,2017-06-01
1366,NaN,2017-08-01
2029,NaN,2017-12-01


In [57]:
df[df['prev_month_rainfall'].isna()] 

,village_code,visit_month,spray_status,total_residents,clinic_tests,total_rdts,avg_distance_to_health_facility_km,incidence_per_1000,positivity_rate,rainfall,prev_month_rainfall,dist_missing
0,7001,2016-12-01,1,1273,0.0,66,7.0,20.98,0.52,4.39,NaN,1
1,7002,2016-12-01,1,1500,14.0,1,7.6,0.00,0.00,4.39,NaN,0
2,7003,2016-12-01,0,3631,14.0,54,1.8,1.90,0.46,4.39,NaN,0
3,7004,2016-12-01,1,5192,14.0,249,0.7,4.86,0.53,4.39,NaN,0
4,7005,2016-12-01,1,2449,14.0,46,0.9,2.83,0.37,4.39,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...
155,7236,2016-12-01,0,341,14.0,6,7.0,34.40,0.67,4.39,NaN,1
326,7237,2017-01-01,0,530,14.0,12,6.0,35.60,0.83,5.64,NaN,0
156,7238,2016-12-01,1,249,14.0,7,7.0,64.52,0.57,4.39,NaN,1
157,7239,2016-12-01,0,394,14.0,30,7.9,109.51,0.57,4.39,NaN,0


In [58]:
df['visit_month'] = pd.to_datetime(df['visit_month'])

fill_values = {
    pd.Timestamp('2016-01-01'): 3.58,
    pd.Timestamp('2017-01-01'): 4.39,
    pd.Timestamp('2017-06-01'): .57,
    pd.Timestamp('2017-12-01'): 6.41,
    pd.Timestamp('2017-02-01'): 5.64,
    pd.Timestamp('2017-05-01'): 4.8,
    pd.Timestamp('2017-08-01'): .82,
    pd.Timestamp('2017-03-01'): 3.42,
    pd.Timestamp('2017-04-01'): 5.78,
}

df['prev_month_rainfall'] = df.apply(
    lambda row: fill_values.get(row['visit_month'], row['prev_month_rainfall']),
    axis=1
)

In [59]:
df['visit_month'].unique()

<DatetimeArray>
['2016-12-01 00:00:00', '2017-01-01 00:00:00', '2017-02-01 00:00:00',
 '2017-03-01 00:00:00', '2017-04-01 00:00:00', '2017-05-01 00:00:00',
 '2017-06-01 00:00:00', '2017-07-01 00:00:00', '2017-08-01 00:00:00',
 '2017-09-01 00:00:00', '2017-10-01 00:00:00', '2017-11-01 00:00:00',
 '2017-12-01 00:00:00', '2018-01-01 00:00:00', '2018-02-01 00:00:00',
 '2018-03-01 00:00:00', '2018-04-01 00:00:00', '2018-05-01 00:00:00',
 '2018-06-01 00:00:00', '2018-07-01 00:00:00', '2018-08-01 00:00:00',
 '2018-09-01 00:00:00', '2018-10-01 00:00:00']
Length: 23, dtype: datetime64[ns]

In [60]:
df[df['prev_month_rainfall'].isna()]['visit_month'].unique()

<DatetimeArray>
['2016-12-01 00:00:00']
Length: 1, dtype: datetime64[ns]

In [61]:
mask = (df['visit_month'] == '2016-12-01') & (df['prev_month_rainfall'].isna())
df.loc[mask, 'prev_month_rainfall'] = 1.55
# That should be the last one

In [62]:
# Let's check if all the NaNs are filled
arr = []
for i in df.columns:
    arr.append(np.sum(df[i].isna()))
arr
# And we are done!

[np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0)]

In [63]:
print(df.columns)
# These are the final and cleaned parameters

Index(['village_code', 'visit_month', 'spray_status', 'total_residents',
       'clinic_tests', 'total_rdts', 'avg_distance_to_health_facility_km',
       'incidence_per_1000', 'positivity_rate', 'rainfall',
       'prev_month_rainfall', 'dist_missing'],
      dtype='object')


In [64]:
len(df[df[avg_dist] == 7][df[df[avg_dist] == 7]['spray_status'] == 1]), len(df[df[avg_dist] == 7])

(470, 1596)

In [65]:
df.to_csv('../CSVs/04_malaria_data_cleaned.csv', index=False)
# PHASE 1: COMPLETE!